In [16]:
import config
import pandas as pd
import modules.capiq as capiq
import modules.gpt as gpt
import textwrap
import modules.sql as sql

In [17]:
excel_path = f'{config.ROOT}/data/testset.xlsx'

db_path = f'{config.ROOT}/data/testset.db'
table_name = 'prompt_outputs'

In [18]:
# Load capiq dataset
df = pd.read_feather(f'{config.ROOT}/data/transcript-detail.feather')

# Keep uhaul calls:
df = df[df['companyid'] == 345588]

In [ ]:
# Specify the list id_list of transcript ids to be examined
id_list = sql.get_transcript_ids(db_path, table_name)
print(f'Number of transcripts in the database: {len(id_list)}')



In [20]:
# Get transcripts
transcript_text = capiq.get_transcripts(transcriptids=id_list, limit=1e7)

In [ ]:
assert len(id_list) == len(transcript_text), "Length of id_list and transcript_text do not match"
print(f"Length of id_list: {len(id_list)}")
print(f"Length of transcript_text: {len(transcript_text)}")

In [ ]:
prompt_name = 'SimplePriceV9'

# Get GPT response:
gpt_response_output = gpt.get_responses(prompt_name, id_list)

In [ ]:
print(gpt_response_output)

In [ ]:
# Make prompt_variables as a list of the keys of the first element of gpt_response_output
response_format_class = gpt.prompts[prompt_name]['response_format']
prompt_variables = list(response_format_class.__fields__.keys())
print(prompt_variables)

In [10]:
# Add columns for current prompt to the database
sql.add_columns_for_prompt(db_path, table_name, prompt_name, prompt_variables)

In [11]:
# Populate the database with the responses
sql.populate_columns_from_response(db_path, table_name, prompt_name, gpt_response_output)

In [ ]:
# This just prints the columns and the contents of the database for quick inspection

import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Query the table schema to see the columns
cursor.execute(f"PRAGMA table_info({table_name})")
columns = cursor.fetchall()

# Print the columns
for column in columns:
    print(column)

# Optionally, query the table data to see the new columns
cursor.execute(f"SELECT * FROM {table_name} LIMIT 5")
rows = cursor.fetchall()

# Print the table data
for row in rows:
    print(row)

# Close the connection
conn.close()



In [13]:
# Export the database for inspection if necessary
csv_path = f'{config.ROOT}/data/testset.csv'
sql.export_table_to_csv(db_path, table_name, csv_path)

In [ ]:
losses = sql.calculate_loss(db_path, table_name)
# For each prompt in losses, print the loss
print('Losses: ')
for prompt, loss in losses.items():
    print(f'{prompt}: {loss}')

In [15]:
# Remove the following columns
# columns_to_remove = ['LeadPriceV1_signal_1', 'LeadPriceV1_score_1', 'LeadPriceV1_reasoning_1', 'LeadPriceV1_signal_2', 'LeadPriceV1_reasoning_2']
#Remove the columns_to_remove from the database
# sql.remove_columns(db_path, table_name, columns_to_remove)
# sql.export_table_to_csv(db_path, table_name, csv_path)